# 01 — Geohazards: Hazard, Risk, and Classification

**Short course:** *Geomorphological Hazards of Slopes* &nbsp;•&nbsp; University of Silesia in Katowice &nbsp;•&nbsp; 2 ECTS

*Lecturer: Ola Fredin*

---

This is the **opening notebook** of the course portfolio. Before we dive into Mohr circles, infinite-slope mechanics, rainfall triggers, and runout statistics, we need a common vocabulary: what *is* a hazard, what *is* a risk, and how do we classify the bewildering variety of slope failures into something we can reason about quantitatively?

The two pillars of this notebook are the **R = H × V × E** decomposition (Varnes 1984, UN-DHA) and the **Hungr et al. (2014) update of the Varnes classification**. Everything that follows in notebooks 02–08 sits inside the box this notebook draws.


## About this notebook

**Learning objectives.** By the end of this notebook the student will be able to:

1. Distinguish hazard, vulnerability, exposure, and risk and combine them quantitatively.
2. Sketch the magnitude–frequency relationship for landslides and explain why small events dominate cumulative impact.
3. Place any given slope failure on the Varnes / Hungr et al. (2014) classification matrix and explain its expected behaviour from that placement.
4. Navigate the rest of the course portfolio — each subsequent notebook addresses one row/column of this matrix.

**Prerequisites.** None. This notebook is the entry point.

> **For your PowerPoint deck.** Three SVG figures land in `figures/`:
> - `risk_decomposition.svg` — bar diagram of R = H × V × E for two contrasting scenarios
> - `magnitude_frequency.svg` — landslide magnitude–frequency power law with the rollover at small sizes
> - `classification_matrix.svg` — the Hungr et al. (2014) movement × material grid


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from style import apply_style, COLORS, save_figure
apply_style()


## 1. Hazard, vulnerability, exposure, risk

The standard decomposition (Varnes 1984; UN International Strategy for Disaster Reduction):

$$
\boxed{\;R \;=\; H \;\times\; V \;\times\; E\;}
\qquad (1)
$$

- **Hazard $H$** — probability that a damaging event of given magnitude occurs at a given location in a given time window. Dimensions: [1/time] or [1/(time·area)]. Determined by physical mechanics: triggers, materials, topography. This is what notebooks 02–08 estimate.
- **Vulnerability $V$** — fraction of value lost given that the event reaches the asset. Dimensionless, $\in [0, 1]$. A wooden house is more vulnerable to a debris flow than a reinforced concrete bridge.
- **Exposure $E$** — value of the asset at risk (people, buildings, infrastructure). Currency units, or counts of people.
- **Risk $R$** — expected loss per unit time. Currency per year.

The decomposition matters because **the three terms come from three different communities**: geologists estimate $H$, engineers estimate $V$, planners estimate $E$. They have to talk to each other for $R$ to make sense.


In [ ]:
# A worked example: a Carpathian village and a Norwegian fjord ferry terminal
# face the same hazard probability but very different exposure and vulnerability.

scenarios = {
    "Carpathian village\n(timber, ~50 houses)": dict(
        H=1e-3,    # 1-in-1000-year shallow translational slide affecting the village
        V=0.70,    # high vulnerability of timber housing to debris flow
        E=2.5e6,   # total replacement value in EUR
    ),
    "Norwegian fjord\nferry terminal":         dict(
        H=1e-3,    # 1-in-1000-year rock-slope failure into the fjord
        V=0.20,    # reinforced concrete, lower vulnerability
        E=5.0e7,   # much higher asset value
    ),
}

# Risk per scenario
fig, ax = plt.subplots(figsize=(8.5, 5.0))
labels = list(scenarios.keys())
risk = [s["H"] * s["V"] * s["E"] for s in scenarios.values()]
xpos = np.arange(len(labels))
bars = ax.bar(xpos, risk, color=[COLORS["soil"], COLORS["water"]], width=0.55,
              edgecolor="black", linewidth=0.5)

# Annotate each bar with the H, V, E breakdown
for i, (lbl, s) in enumerate(scenarios.items()):
    r = s["H"] * s["V"] * s["E"]
    ax.text(i, r * 1.05,
            fr"$H$ = {s['H']:.0e}, $V$ = {s['V']:.2f}, $E$ = {s['E']:.1e} EUR",
            ha="center", fontsize=11)
    ax.text(i, r * 0.5,
            fr"$R$ = {r:,.0f}  EUR/yr",
            ha="center", va="center", fontsize=14, color="white", fontweight="bold")

ax.set_xticks(xpos); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("expected loss  $R$  [EUR / yr]")
ax.set_title("Risk decomposition  $R = H \times V \times E$  for two contrasting assets")
ax.set_ylim(0, max(risk) * 1.30)
save_figure(fig, "risk_decomposition")
plt.show()

print("Same hazard probability, very different risk:")
for lbl, s in scenarios.items():
    r = s["H"] * s["V"] * s["E"]
    print(f"  {lbl.replace(chr(10), ' ')}:  R = {r:,.0f} EUR/yr")


## 2. Magnitude–frequency: the long tail

Landslide populations obey a remarkably robust statistical law: above a *rollover scale* of ~10²–10³ m², the cumulative number $N$ of events with area larger than $A$ scales as a power law,

$$
N(A) \;\propto\; A^{-b}, \qquad b \approx 1.0\,\text{–}\,1.5
\qquad (2)
$$

(Malamud et al. 2004, summarising Italian, US, and Guatemalan inventories.) Below the rollover scale, small events are systematically *underrepresented* — partly because they are physically suppressed by soil cohesion and root strength, partly because small slides are under-mapped.

Two practical consequences:

- **Big events dominate the elevation budget** (sediment delivery, erosion). One $10^7$ m³ event delivers as much sediment as $10^6$ events of $10$ m³.
- **Small events dominate the human exposure budget** (people affected, property damage). They occur orders of magnitude more often and are clustered near roads, settlements, and farms.

A regional hazard analysis therefore needs **both ends of the distribution**: rare extreme events for catastrophic risk, frequent small events for routine planning.


In [ ]:
# Synthetic landslide inventory: power-law with rollover.
rng = np.random.default_rng(123)
n_events = 5000
A_min, A_rollover, A_max = 1.0, 100.0, 1e7
b_exponent = 1.3

# Sample from power-law for large events; suppress below rollover.
u = rng.random(n_events)
A_powerlaw = A_rollover * (1.0 - u)**(-1.0 / b_exponent)
# Apply rollover suppression (lognormal-like below A_rollover)
suppress = np.where(A_powerlaw < A_rollover,
                    A_powerlaw * np.exp(rng.normal(0, 0.8, n_events)),
                    A_powerlaw)
A_obs = np.clip(suppress, A_min, A_max)

# Cumulative N(>=A)
A_sorted = np.sort(A_obs)
N_ge = len(A_sorted) - np.arange(len(A_sorted))

fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.loglog(A_sorted, N_ge, ".", ms=3, color=COLORS["fail"], alpha=0.6,
          label="synthetic regional inventory")
A_line = np.logspace(np.log10(A_rollover), np.log10(A_max), 100)
N_line = 1.2 * len(A_sorted) * (A_line / A_rollover)**(-b_exponent)
ax.loglog(A_line, N_line, color=COLORS["rock"], lw=2.0, ls="--",
          label=fr"power law  $N \propto A^{{-{b_exponent:.1f}}}$")
ax.axvline(A_rollover, color=COLORS["accent"], lw=1.2, ls=":")
ax.text(A_rollover * 1.4, 5, "rollover\n(small slides suppressed)",
        color=COLORS["accent"], fontsize=11, va="bottom")
ax.set_xlabel(r"landslide area  $A$  [m$^2$]")
ax.set_ylabel(r"cumulative count  $N(\geq A)$")
ax.set_title("Magnitude–frequency distribution of a landslide population (synthetic)")
ax.legend(loc="lower left")
ax.grid(True, which="both", alpha=0.4)
save_figure(fig, "magnitude_frequency")
plt.show()


## 3. The Hungr et al. (2014) classification

The 1978 Varnes classification — five movement types (falls, topples, slides, flows, spreads) × three materials (rock, debris, earth) — has dominated landslide vocabulary for forty years. Hungr et al. (2014) updated it to remove ambiguities, harmonise national usages, and absorb new process knowledge.

The matrix below shows the canonical entries. The processes in **bold** are the ones treated explicitly in this course portfolio.

|              | Rock                          | Debris (mixed)                              | Soil / earth                          | Ice / snow             |
|--------------|-------------------------------|---------------------------------------------|---------------------------------------|------------------------|
| **Fall**     | **rock fall** (nb 06)         | debris fall                                 | soil/earth fall                       | ice fall               |
| **Topple**   | rock topple                   | debris topple                               | earth topple                          | —                      |
| **Slide**    | rock slide                    | **shallow translational** (nb 04)           | earthflow, slow earth slide           | —                      |
| **Flow**     | rock avalanche                | **debris flow** (nb 07), debris avalanche   | mudflow, sand-/silt liquefaction      | snow avalanche         |
| **Spread**   | rock-mass spread              | debris spread                               | liquefaction spread                   | —                      |
| **Complex**  | composite rockslide–avalanche | complex debris slide–flow                   | composite earth slide–flow            | —                      |

Two things to flag for exam purposes:

- **"Debris" is a material, not a process.** It denotes a poorly sorted mixture of rock fragments, soil, and organic matter — typical of glacial till, weathered flysch, or earlier landslide deposits. The same material can host a slide, a flow, a fall, or a spread.
- **Movement types can transition.** A shallow translational *slide* in saturated colluvium often turns into a debris *flow* as it accelerates and entrains water and bed material. Hungr et al. give "complex" labels for the common transitions. This is exactly the slide → flow chain that motivates notebooks 04 → 05 → 07.


In [ ]:
# Plot the classification as a colour-coded grid, with treated processes highlighted.
movement_types = ["Fall", "Topple", "Slide", "Flow", "Spread", "Complex"]
materials = ["Rock", "Debris\n(mixed)", "Soil / earth", "Ice / snow"]

# 0 = not covered by this course, 1 = covered. Map indexed (row=movement, col=material)
covered = np.array([
    [0, 1, 0, 0],   # Fall (rock fall = nb 06)
    [0, 0, 0, 0],   # Topple
    [0, 1, 0, 0],   # Slide (shallow translational = nb 04)
    [0, 1, 0, 0],   # Flow (debris flow = nb 07)
    [0, 0, 0, 0],   # Spread
    [0, 1, 0, 0],   # Complex
])

# Cell labels
labels = [
    ["rock fall",        "debris fall",         "soil fall",       "ice fall"],
    ["rock topple",      "debris topple",       "earth topple",    "—"],
    ["rock slide",       "shallow\ntranslational", "earthflow",       "—"],
    ["rock avalanche",   "debris flow,\ndebris avalanche", "mudflow,\nliquefaction",  "snow avalanche"],
    ["rock-mass\nspread", "debris spread",       "liquefaction\nspread", "—"],
    ["rockslide–\navalanche", "debris\nslide–flow",  "earth\nslide–flow",  "—"],
]

fig, ax = plt.subplots(figsize=(11.5, 6.8))
for i in range(len(movement_types)):
    for j in range(len(materials)):
        is_covered = covered[i, j] == 1
        bg = COLORS["accent"] if is_covered else "#F0F0F0"
        rect = Rectangle((j, len(movement_types) - 1 - i), 1, 1,
                         facecolor=bg, edgecolor="black", linewidth=0.8, alpha=0.7)
        ax.add_patch(rect)
        weight = "bold" if is_covered else "normal"
        ax.text(j + 0.5, len(movement_types) - 1 - i + 0.5, labels[i][j],
                ha="center", va="center", fontsize=10.5,
                color="black", fontweight=weight)

ax.set_xlim(0, len(materials))
ax.set_ylim(0, len(movement_types))
ax.set_xticks(np.arange(len(materials)) + 0.5)
ax.set_xticklabels(materials, fontsize=12)
ax.set_yticks(np.arange(len(movement_types)) + 0.5)
ax.set_yticklabels(reversed(movement_types), fontsize=12)
ax.set_aspect("equal")
ax.set_title("Hungr et al. (2014) update of the Varnes classification\n"
             "(orange cells are treated explicitly in this course portfolio)")
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
save_figure(fig, "classification_matrix")
plt.show()


## 4. Five reference cases worth knowing

Every introduction to slope hazards should leave the student with a handful of touchstone events whose facts they can call to mind on demand. Five well-documented examples spanning the scale and process range:

| Event | Year | Type (Hungr) | Volume | Fatalities | Note |
|---|---|---|---|---|---|
| **Vajont, Italy** | 1963 | composite rockslide → impulse wave | $2.7 \times 10^8$ m³ | ~2 000 | reservoir filling raised $u$; rotational failure displaced water over the dam |
| **Storegga, Norway (offshore)** | ~8 200 BP | submarine debris avalanche | $3 \times 10^{12}$ m³ | (prehistoric) | largest known submarine slide; tsunami reached the Scottish coast |
| **Frank Slide, Canada** | 1903 | rock avalanche | $3.0 \times 10^7$ m³ | ~70 | runout 4 km; alpha ≈ 14° |
| **Aberfan, Wales** | 1966 | flowslide from a coal-tip embankment | $1.5 \times 10^5$ m³ | 144 (mostly children) | triggered by sustained rainfall on a saturated waste pile |
| **Małopolska floods, Poland** | 2010 | swarm of shallow translational slides → debris flows | $10^7$ m³ cumulative | a few dozen total | thousands of slides; trigger was 200+ mm rain over four days (cf. notebook 05) |

Each event spotlights one of the mechanisms in the rest of the course portfolio: Vajont = pore-pressure triggering (nb 02); Storegga = submarine analogue of debris-flow runout statistics (nb 07); Frank = alpha-line / rockfall runout in hard rock (nb 06); Aberfan = shallow translational slide in saturated debris (nb 04); 2010 floods = rainfall-driven mass triggering across a catchment (nb 05).


## 5. National versus international classifications

The classification in §3 is the *international* consensus, used in Landslides, NHESS, Geomorphology, and similar journals. Several national systems differ in important ways — the Polish SOPO inventory and the Vietnamese national classification are the ones the course coordinator-Faculty will most often encounter:

- **Polish SOPO (System Osłony Przeciwosuwiskowej).** Captures landslides in the Polish Carpathians with a national-scale GIS database. Uses a six-category typology that maps roughly to the Hungr et al. scheme but with explicit attention to **complex slides** typical of Carpathian flysch. Reaches different counts to the Hungr framework for the same Carpathian terrain — a calibration step is needed when comparing.
- **Vietnamese practice.** "Debris avalanche" in Vietnamese landslide papers commonly refers to what European usage would call a "*complex* debris slide–flow" — a confined channelised event that initiated as a slide. A direct translation without clarification can lead to systematic miscounting.

The general lesson: **data harmonisation is a precondition** for any cross-border or global landslide statistics. The Fidan et al. (2024) global compilation (cited in the course bibliography) addresses this explicitly by re-classifying national contributions into a common Hungr-based vocabulary before regression analysis.


## 6. How to use this course portfolio

The notebook portfolio is structured as a **layered build**: each notebook adds one mechanical concept, with the running Carpathian flysch slope as the shared example. Suggested reading order:

| #  | Topic                          | What you'll learn |
|----|--------------------------------|-------------------|
| 00 | template                       | the structural conventions used throughout |
| 01 | *(you are here)*               | vocabulary and classification |
| 02 | effective stress               | the σ = σ' + u foundation |
| 03 | Mohr–Coulomb                   | shear strength + failure envelope |
| 04 | infinite slope                 | FS equation, saturation, critical depth |
| 05 | rainfall triggers              | Iverson infiltration + ID thresholds + FS(t) |
| 06 | rockfall trajectory            | shadow angle + lumped-mass ballistic + Monte Carlo |
| 07 | debris flow runout             | Iverson/Rickenmann scaling + α–β + Monte Carlo |
| 08 | LiDAR + hillshade              | DEMs, hillshade, slope/aspect, landslide mapping |

The chain 02 → 03 → 04 → 05 is **strongly linear** — each notebook depends on the previous. Notebooks 06, 07, 08 are independent of each other and can be visited in any order after 04.

For lecture pacing in 16 hours:

- **Day 1** (≈ 6 h):  Lecture on classification + hazard/risk → notebook 01, then 02, then 03.
- **Day 2** (≈ 6 h):  Lecture on slope stability + triggers → notebooks 04, 05.
- **Day 3** (≈ 4 h):  Lecture on runout + mapping → notebooks 06 *or* 07 (pick one), then 08 as the hands-on closer.


## Take-aways

- Risk is decomposed as $R = H \times V \times E$. Geologists estimate $H$; the other terms come from engineers and planners. All three are needed.
- Landslide populations are power-law-distributed above a small-event rollover. Big events dominate the sediment budget; small events dominate the exposure budget.
- The Hungr et al. (2014) update of Varnes is the international consensus classification: movement × material with a sixth "complex" row for transitions.
- Many real slope failures change type as they evolve — a slide becomes a flow, a fall fragments into an avalanche. Notebooks 04–07 trace these specific transitions mechanically.
- National classifications (Polish SOPO, Vietnamese practice) differ from the international scheme in non-trivial ways. Cross-border comparisons need explicit harmonisation.


## Questions for the exam

1. Define hazard, vulnerability, and exposure in your own words. Construct an example where two assets face the same hazard but have wildly different risk.
2. Sketch the cumulative magnitude–frequency curve for a regional landslide inventory. Identify the rollover and explain its origin. Where on the curve do the biggest sediment-budget contributors sit? Where do the biggest exposure contributors sit?
3. A shallow failure on a forested Carpathian slope began as a translational slide and reached the village 800 m downslope as a debris flow. Place this event on the Hungr et al. classification matrix and identify the row that captures the transition.
4. Why is "debris avalanche" an ambiguous term across the European and Vietnamese literature? What harmonisation step would you do before using the two national inventories in a comparative analysis?


## References

- Varnes, D. J. (1978). *Slope movement types and processes.* In *Landslides: Analysis and Control*, Transportation Research Board Special Report 176, 11–33.
- Varnes, D. J. (1984). *Landslide hazard zonation: a review of principles and practice.* UNESCO Natural Hazards Series, 3.
- Hungr, O., Leroueil, S., & Picarelli, L. (2014). *The Varnes classification of landslide types, an update.* Landslides, 11(2), 167–194. https://doi.org/10.1007/s10346-013-0436-y
- Malamud, B. D., Turcotte, D. L., Guzzetti, F., & Reichenbach, P. (2004). *Landslide inventories and their statistical properties.* Earth Surface Processes and Landforms, 29(6), 687–711.
- Fidan, S., et al. (2024). *Global landslide overview: frequency, distribution, and socioeconomic impact.* Natural Hazards. https://doi.org/10.1007/s11069-024-06487-3
- Korup, O. & Paněk, T. (2025). *Communications Earth & Environment.* https://doi.org/10.1038/s43247-025-02614-5
- Polish SOPO portal: https://geoportal.gov.pl
